# Sekiro: Shadows Die Twice - Boss Detection
### Notebook 2 of 2: Extended Track (YOLO)

This notebook fine-tunes and evaluates the boss detector that feeds the
*Extended Track* of the assistant: a screenshot goes in, a boss name comes out,
and that name becomes a metadata filter on the Core Track's retrieval.

It covers Sections **5.5.1 - 5.5.7** of the spec:

| Section | What it is |
|---|---|
| 5.5.1 | Choosing the boss classes |
| 5.5.2 | Footage and recording sessions |
| 5.5.3 | Frame extraction |
| 5.5.4 | Filtering and selection |
| 5.5.5 | Auto-labelling: measured and rejected |
| 5.5.6 | Training configuration |
| 5.5.7 | Evaluation |

**Environment.** The training cells need the CUDA venv, not the project venv:

```
D:/Mustafa/programming/.venv-gpu/Scripts/python.exe
```

**A note on reproducibility.** The two image datasets
(`data/images/dataset/` and `data/images/dataset5/`) are deliberately
**gitignored** - together they are ~1.5 GB, and the first is a third-party
Roboflow export rather than our own artifact. So a fresh clone cannot re-run the
training cells as-is. Every data-dependent cell below detects a missing input and
prints the figure that was recorded when the pipeline did run, so the notebook
still tells the whole story and still executes top to bottom.

---
## 0. Setup

Paths resolve from the notebook's own location, so "Restart & Run All" works
from any launch directory.

In [ ]:
import json
import shutil
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA = PROJECT_ROOT / "data" / "images"
SOURCE_DATASET = DATA / "dataset"          # 40-class Roboflow export  (gitignored)
DATASET = DATA / "dataset5"                # merged 5-class dataset    (gitignored)
RUNS = PROJECT_ROOT / "runs"               # ultralytics output        (gitignored)
WEIGHTS = RUNS / "boss5_v1" / "weights" / "best.pt"

# Scratch workspace that produced the dataset. Lives beside the repo, not in it,
# because it holds ~300 MB of extracted frames. Absent on a fresh clone.
SCRATCH = PROJECT_ROOT.parent / "_yolo_rebuild"
EVAL_JSON = SCRATCH / "eval_test.json"
CLIP_JSON = SCRATCH / "clipped_labels.json"

# The five classes, in the contiguous id order build_dataset.py writes.
CLASSES = ["corrupted_monk", "divine_dragon", "genichiro", "guardian_ape", "owl"]

# Original class ids inside the 40-class Roboflow export.
SOURCE_IDS = {"corrupted_monk": 5, "divine_dragon": 7, "genichiro": 12,
              "guardian_ape": 14, "owl": 30}

TRAIN_IMGSZ = 640
CONF_THRESHOLD = 0.5

# Flip to True to retrain from scratch. Left False so the notebook is cheap to
# re-run end to end; the committed weights are the ones the backend loads.
RETRAIN = False

print("project root :", PROJECT_ROOT)
print("scratch dir  :", SCRATCH, "(present)" if SCRATCH.exists() else "(absent - recorded figures will be shown)")

---
## 5.5.1 - Choosing the boss classes

The spec asks for **4-5 visually distinct bosses, not 10+**, each with enough
labelled instances to train on. The source export carries **40** classes, which
is far past that - so the first job was deciding which five survive.

Two questions decide it, and they point the same way:

1. **Is the class visually distinguishable?** A detector cannot separate two
   bosses that share a silhouette, a colour and a backdrop.
2. **Is the class sampled enough?** Spec 5.5.4 sets a floor of 60-100 labelled
   instances per class.

The cell below counts what the source export actually contains, per class.

In [ ]:
def count_instances(dataset_dir: Path) -> Counter:
    """Count label instances per class id across all splits."""
    counts = Counter()
    for split in ("train", "valid", "test"):
        for label in (dataset_dir / split / "labels").glob("*.txt"):
            for line in label.read_text(errors="replace").splitlines():
                parts = line.split()
                if len(parts) == 5:
                    counts[int(parts[0])] += 1
    return counts


if SOURCE_DATASET.exists():
    source_counts = count_instances(SOURCE_DATASET)
    n_images = sum(len(list((SOURCE_DATASET / s / "images").glob("*.jpg")))
                   for s in ("train", "valid", "test"))
    totals = sorted(source_counts.values(), reverse=True)
    above = sum(1 for v in totals if v >= 60)

    print(f"source export: {n_images} images, {len(source_counts)} classes\n")
    print(f"  instances per class : mean {sum(totals)/len(totals):.1f}, "
          f"max {totals[0]}, min {totals[-1]}")
    print(f"  imbalance           : {totals[0]}:1")
    print(f"  classes >= 60 (usable)  : {above}")
    print(f"  classes <  60 (starved) : {len(totals) - above}")
    print()
    print("  the five the spec names, before any of our own labelling:")
    for name, sid in SOURCE_IDS.items():
        n = source_counts[sid]
        flag = "ok" if n >= 60 else "BELOW the 60 floor"
        print(f"    {name:<16} {n:>4}  {flag}")
else:
    print("source export absent (gitignored). Recorded figures:")
    print("  1813 images, 40 classes, mean 42.0 instances/class")
    print("  imbalance 317:1 -- 7 classes clear the 60 floor, 33 fall short")
    print("  spec classes: corrupted_monk 48, divine_dragon 21, "
          "genichiro 119, guardian_ape 97, owl 317")

**Reading the result.** Only **7 of 40** classes clear the 60-instance floor,
and the mean sits at **42 per class** - the export is a long tail of bosses seen
once or twice. With a 317:1 imbalance, a model trained on all 40 would spend its
capacity on the head and never learn the tail.

The five chosen are the ones the spec names, and they are also the right five on
the merits: each has a distinct silhouette (a tall robed figure, a serpentine
dragon, a swordsman, a white ape, a great owl) and a distinct arena. Two of them
- `corrupted_monk` (48) and `divine_dragon` (21) - are **below the floor in the
source export**, so those two are precisely where the hand-labelling effort in
5.5.3-5.5.5 went.

---
## 5.5.2 - Footage and recording sessions

Spec 5.5.2 asks for footage spanning **at least two recording sessions**, so the
detector does not overfit one playthrough's lighting, HUD state or camera habits.

What we have is one continuous capture:

In [ ]:
import cv2

FOOTAGE = DATA / "raw_footage"

if FOOTAGE.exists() and any(FOOTAGE.glob("*.mp4")):
    for video in sorted(FOOTAGE.glob("*.mp4")):
        cap = cv2.VideoCapture(str(video))
        fps = cap.get(cv2.CAP_PROP_FPS)
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        secs = n / fps
        print(f"{video.name}")
        print(f"  {w}x{h} @ {fps:.0f} fps, {n} frames, "
              f"{int(secs//3600)}h {int(secs%3600//60)}m {int(secs%60)}s")
else:
    print("footage absent (gitignored). Recorded: videoplayback.mp4, "
          "1920x1080 @ 60 fps, 353594 frames, 1h 38m")

### Honest limitation to carry into 5.5.7

**All of our own labelling comes from this single session.** The existing
Roboflow export does span more than one playthrough - that is the only way to
explain two files sharing a timestamp yet showing genuinely different frames,
each with its own correct label (see 5.5.5) - but the 275 frames *we* labelled by
hand are all from the video above.

So for `corrupted_monk` and `divine_dragon`, the two classes that depend on our
labels to clear the sufficiency floor, the two-session requirement is **not**
independently satisfied. This is stated again in the evaluation section rather
than left to be discovered during a demo.

---
## 5.5.3 - Frame extraction

Frames are pulled at **4 fps**, not every frame. At 60 fps a three-second sword
swing yields 180 near-identical images; at 4 fps it yields 12, still more than
enough to cover the pose range without a dataset that is 95% redundancy.

The extraction script is `extract_frames.py` in the scratch workspace. Its
output is what the next cell summarises.

In [ ]:
if SCRATCH.exists() and (SCRATCH / "extracted").exists():
    extracted = sorted((SCRATCH / "extracted").glob("*.jpg"))
    print(f"extracted frames : {len(extracted)}")
    print(f"on-disk size     : "
          f"{sum(f.stat().st_size for f in extracted)/1e6:.0f} MB")
    print(f"source video     : 353594 frames @ 60 fps -> sampled at 4 fps")
    print(f"                   = {353594/15:.0f} frames expected")
else:
    print("extracted/ absent (scratch workspace, not committed). Recorded:")
    print("  1471 frames at 4 fps, 240 MB; 1419 visually distinct")

Of the **1471** frames written, **1419** are visually distinct - 52 are duplicate
frames the encoder emitted around scene cuts. Frame identity is carried in the
filename as `frame_<seconds>.<tenths>s`, which becomes the key that keeps frames
from the same instant out of two different splits later on.

---
## 5.5.4 - Filtering and selection

Two filters run over the 1471 extracted frames.

**Filter 1 - distinctness.** Drop the 52 exact duplicates, leaving 1419.

**Filter 2 - a balanced selection.** The aim is not "label everything", it is
"label until each class clears the floor". `owl`, `genichiro` and `guardian_ape`
already clear 60 in the existing export, so the labelling budget went almost
entirely to the two starved classes:

| Class | In export | Selected to label | Why |
|---|---|---|---|
| `divine_dragon` | 21 | **110** | far below the floor; needs the most |
| `corrupted_monk` | 48 | **85** | below the floor |
| `guardian_ape` | 97 | 50 | already sufficient; top-up only |
| `genichiro` | 119 | 30 | already sufficient; top-up only |
| `owl` | 317 | 0 | already the head class |
| | | **275** | |

Frames were chosen **blind** - no detector existed yet - so some show the boss
small or at distance. Those were skipped during labelling rather than given a
low-quality box; the tool moves to the next frame with one keypress.

In [ ]:
if SCRATCH.exists() and (SCRATCH / "to_label").exists():
    planned = {"divine_dragon": 110, "corrupted_monk": 85,
               "guardian_ape": 50, "genichiro": 30}
    print(f"{'class':<16}{'selected':>9}{'usable labels':>15}")
    for name, n in planned.items():
        labels = SCRATCH / "to_label" / name / "labels"
        have = sum(1 for p in labels.glob("*.txt")
                   if p.read_text().strip()) if labels.exists() else 0
        print(f"{name:<16}{n:>9}{have:>15}")
    print(f"\ntotal selected: {sum(planned.values())}")
else:
    print("to_label/ absent (scratch workspace, not committed). Recorded:")
    print("  divine_dragon 110 selected / 74 usable")
    print("  corrupted_monk  85 selected / 63 usable")
    print("  guardian_ape    50 selected /  0 usable (never labelled)")
    print("  genichiro       30 selected /  0 usable (never labelled)")
    print("  total 275 selected")

### A labelling-tool bug worth recording

Dragging a box past the image boundary stored a box hanging off the frame. A
detector can never regress to that, and ground truth should describe what is
visible, so `fix_clipped_labels.py` clamps those boxes to `[0, 1]`.

In [ ]:
if CLIP_JSON.exists():
    clipped = json.loads(CLIP_JSON.read_text())
    by_class = Counter(e["class"] for e in clipped)
    print(f"boxes clamped to the frame edge: {len(clipped)}")
    for name, n in by_class.most_common():
        print(f"  {name:<16}{n}")
    print("\noriginals preserved in to_label/<class>/labels_preclip/, so the")
    print("change is reversible; every before/after pair is in clipped_labels.json")
else:
    print("clipped_labels.json absent. Recorded: 22 boxes clamped "
          "(17 divine_dragon, 5 corrupted_monk), mostly 2-6 px of top-edge")
    print("overshoot where the boss is cut off by the viewport.")

---
## 5.5.5 - Auto-labelling: measured and rejected

The obvious shortcut is to label one frame by hand and propagate the box to its
neighbours, since they are only a fraction of a second apart. That was measured
before it was trusted, and it does not work.

The decisive number is the **ceiling**: take the ground-truth boxes on two frames
one second apart and measure how well the first predicts the second. That is the
best any propagation method could possibly score, and it is low - Sekiro bosses
cover ground fast.

In [ ]:
# Recorded by validate_propagation.py and validate_tracking.py in the scratch
# workspace. Both need the source footage, so they are not re-run here.
methods = [
    ("Linear interpolation of boxes",       0.463, 0.12),
    ("Lucas-Kanade optical-flow tracking",  0.335, 0.03),
    ("Ceiling: ground truth, 1 s apart",    0.430, None),
]

print(f"{'method':<38}{'mean IoU':>10}{'frames >= 0.7':>16}")
for name, iou, frac in methods:
    frac_s = "-" if frac is None else f"{frac:.0%}"
    print(f"{name:<38}{iou:>10.3f}{frac_s:>16}")

print("\nInterpolation scores 0.463 against a ceiling of 0.430 -- it is already")
print("at the limit, and 88% of propagated boxes still land below IoU 0.7.")
print("The boxes cannot be predicted; they have to be drawn by hand.")

### 5.5.5.1 - Merging into one 5-class dataset

`build_dataset.py` merges the 40-class export with the 275 hand-drawn frames.
Three decisions in it are worth stating, because each could reasonably have gone
the other way:

**Frames containing only non-spec bosses are dropped, not kept as background.**
An image showing, say, `lady_butterfly` would end up with an empty label file -
which teaches the model that "no box here" is correct for that visual pattern.
That is a false negative by construction. **1070** such frames were dropped.
Genuinely empty frames (menus, scenery, death screens) *are* kept, as real
negatives.

**New frames are rendered to 640x640 to match the existing ones.** The existing
export is 640x640 **stretched** from a 1362x767 stream viewport, not a square
crop. Leaving the new frames at native aspect would train the model on two
different geometries at once. A resize preserves normalised coordinates, so the
hand-drawn labels transfer unchanged. Inference must therefore stretch to
640x640 as well, not letterbox - see 5.5.8.

**The new frames split 85/15 by timestamp**, not by random shuffle, so the
holdout is a later slice of each fight rather than a random scattering of it.

In [ ]:
if DATASET.exists():
    print(f"{'split':<8}{'images':>8}{'labels':>8}{'background':>12}")
    for split in ("train", "valid", "test"):
        imgs = sorted((DATASET / split / "images").glob("*.jpg"))
        labels = DATASET / split / "labels"
        bg = sum(1 for i in imgs
                 if not (labels / f"{i.stem}.txt").exists()
                 or not (labels / f"{i.stem}.txt").read_text().strip())
        print(f"{split:<8}{len(imgs):>8}{len(imgs)-bg:>8}{bg:>12}")

    counts = count_instances(DATASET)
    print(f"\n{'class':<16}{'id':>4}{'instances':>11}{'vs 60 floor':>14}")
    for cid, name in enumerate(CLASSES):
        n = counts[cid]
        print(f"{name:<16}{cid:>4}{n:>11}{'ok' if n >= 60 else 'BELOW':>14}")

    totals = sorted(counts.values(), reverse=True)
    print(f"\nimbalance now: {totals[0]}:{totals[-1]} "
          f"(was 317:1 in the source export)")
else:
    print("dataset5 absent (gitignored). Recorded:")
    print("  train 725 images / 612 labelled / 113 background")
    print("  valid 108 images /  93 labelled /  15 background")
    print("  test   38 images /  38 labelled /   0 background")
    print("  1070 frames dropped for containing only non-spec bosses")
    print()
    print("  corrupted_monk   0  104   ok      (48 in export, +56 hand-drawn)")
    print("  divine_dragon    1   93   ok      (21 in export, +72 hand-drawn)")
    print("  genichiro        2  119   ok")
    print("  guardian_ape     3   97   ok")
    print("  owl              4  317   ok")
    print("  imbalance 317:93 = 3.4:1")

**Result.** All five classes now clear the 60-instance floor, and the imbalance
falls from **317:1 to 3.4:1**. `divine_dragon` goes from 21 to 93 - a 4.4x
increase - which is the single change that made this dataset trainable at all.

### Duplicate leakage, found and removed

Twelve frames turned out to exist in **both** the Roboflow export and our new
hand-drawn bundle - the same video instant, labelled twice, with boxes that
agreed only at mean IoU 0.505. Three of the twelve sat in the export's **valid**
split, so adding our copies to train would have been textbook train/validation
leakage.

Our copies were moved out to `removed_duplicates/`, with a manifest listing every
one. The cost was 9 hand-drawn labels (`divine_dragon` 74 -> 72,
`corrupted_monk` 63 -> 56). The alternative - dropping the export's copies -
would have cost 12 labels *and* disturbed a valid split built by someone else.

---
## 5.5.6 - Training

**Starting point: `yolov8n.pt`, the pretrained nano checkpoint.** The spec
requires this and the reason is arithmetic - 871 images cannot teach a network to
see from scratch, but they can fine-tune one that already knows what edges and
textures look like.

| Setting | Value | Reason |
|---|---|---|
| Model | `yolov8n.pt` | nano: fastest to train, smallest to ship, adequate for 5 large objects |
| Epochs | 100 | upper bound; early stopping decides the real count |
| Patience | 25 | the train set is 725 images - it overfits long before 100 epochs |
| Image size | 640 | matches the dataset, which is already 640x640 |
| Batch | 8 | RTX 3050 Ti has 4 GB; batch 16 at 640 does not fit |
| Seed | 0 | so the run is repeatable |
| Optimiser | SGD (ultralytics default) | |

Training ran on the GPU venv against an RTX 3050 Ti.

In [ ]:
args_file = RUNS / "boss5_v1" / "args.yaml"
results_csv = RUNS / "boss5_v1" / "results.csv"

if RETRAIN:
    from ultralytics import YOLO
    YOLO("yolov8n.pt").train(
        data=str(DATASET / "data.yaml"), epochs=100, imgsz=TRAIN_IMGSZ,
        batch=8, patience=25, seed=0, device=0,
        project=str(RUNS), name="boss5_v1", exist_ok=True,
        workers=2, plots=True, val=True,
    )

if results_csv.exists():
    history = pd.read_csv(results_csv)
    history.columns = [c.strip() for c in history.columns]
    best = history.loc[history["metrics/mAP50(B)"].idxmax()]
    print(f"epochs completed : {len(history)}")
    print(f"best epoch       : {int(best['epoch'])}")
    print(f"  train box loss : {best['train/box_loss']:.4f}")
    print(f"  precision      : {best['metrics/precision(B)']:.4f}")
    print(f"  recall         : {best['metrics/recall(B)']:.4f}")
    print(f"  mAP50          : {best['metrics/mAP50(B)']:.4f}")
    print(f"  mAP50-95       : {best['metrics/mAP50-95(B)']:.4f}")
else:
    print("no run found at runs/boss5_v1 (gitignored). Set RETRAIN=True to")
    print("rebuild it -- needs data/images/dataset5/ and the CUDA venv.")

In [ ]:
if results_csv.exists():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    history.plot(x="epoch", y=["train/box_loss", "train/cls_loss", "train/dfl_loss"],
                 ax=axes[0], title="Training losses")
    history.plot(x="epoch",
                 y=["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)"],
                 ax=axes[1], title="Validation metrics")
    axes[1].axhline(0.5, ls=":", c="grey", label="mAP50 = 0.5")
    for ax in axes:
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("training curve unavailable without runs/boss5_v1/results.csv")

**Reading the curves.** Precision climbs fastest and settles high - the model
quickly stops firing on scenery. Recall is the slower one, which is what a
725-image train set predicts: the model is conservative about a boss it has seen
in only a handful of poses. The loss curves flatten well before the epoch cap,
which is why early stopping rather than a fixed 100 epochs is the right setting
here.

---
## 5.5.7 - Evaluation

Measured on the **test split**, which is held out from both training and the
validation pass that early stopping uses - so these numbers are not the ones
already visible in `results.csv`.

In [ ]:
def show_block(title, note, block):
    o = block["overall"]
    print(f"\n=== {title} ===")
    print(f"{note}")
    print(f"{block['images']} images, {block['instances']} instances, "
          f"IoU {block['iou_threshold']}")
    print(f"overall  P {o['precision']:.4f}   R {o['recall']:.4f}   "
          f"mAP50 {o['mAP50']:.4f}   mAP50-95 {o['mAP50-95']:.4f}\n")
    print(f"{'class':<16}{'inst':>6}{'P':>9}{'R':>9}{'mAP50':>9}{'mAP50-95':>10}   note")
    for name, m in block["per_class"].items():
        flag = "n too small to trust" if m["instances"] < 10 else ""
        print(f"{name:<16}{m['instances']:>6}{m['precision']:>9.4f}"
              f"{m['recall']:>9.4f}{m['mAP50']:>9.4f}{m['mAP50-95']:>10.4f}   {flag}")


report = json.loads(EVAL_JSON.read_text()) if EVAL_JSON.exists() else None

if report:
    show_block("TEST split (clean holdout)",
               "Never used for training or early stopping -- but small.",
               report["test"])
    show_block("VALID split (supporting only)",
               "Usable per-class n, but this was the early-stopping signal.",
               report["valid"])
else:
    print("no eval_test.json (scratch workspace, not committed).")
    print("Regenerate with evaluate.py, or set RETRAIN=True and re-run the")
    print("training cells, then re-run evaluate.py.")

In [ ]:
if report:
    names = list(report["valid"]["per_class"])
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    for ax, split in zip(axes, ("test", "valid")):
        block = report[split]
        cls = list(block["per_class"])
        x = range(len(cls))
        w = 0.38
        ax.bar([i - w/2 for i in x],
               [block["per_class"][n]["precision"] for n in cls], w, label="precision")
        ax.bar([i + w/2 for i in x],
               [block["per_class"][n]["recall"] for n in cls], w, label="recall")
        ax.set_xticks(list(x))
        ax.set_xticklabels(cls, rotation=20, ha="right", fontsize=8)
        ax.set_ylim(0, 1)
        ax.axhline(0.5, ls=":", c="grey")
        ax.set_title(f"{split} split ({block['instances']} instances)")
        ax.grid(axis="y", alpha=0.3)
    axes[0].legend()
    fig.suptitle("Per-class precision and recall at IoU 0.50")
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 4))
    cls = list(report["valid"]["per_class"])
    ax.barh(cls, [report["valid"]["per_class"][n]["mAP50"] for n in cls], label="mAP50")
    ax.barh(cls, [report["valid"]["per_class"][n]["mAP50-95"] for n in cls],
            label="mAP50-95", alpha=0.75)
    ax.set_xlim(0, 1)
    ax.axvline(0.5, ls=":", c="grey")
    ax.set_title("Per-class mAP, valid split")
    ax.legend()
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

### mAP is not the operating point

mAP is **threshold-free**: it sweeps every confidence cut from 0 to 1 and measures
how well the predictions are *ranked*. A detector can therefore post a respectable
mAP while almost never clearing the confidence threshold the service actually
uses. Whether that happened here is a question about the deployed cut, not about
mAP, so it is measured separately - matching greedily one-to-one at IoU 0.5, so a
class cannot inflate its own recall by firing many boxes at one boss.

In [ ]:
SWEEP = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60]

# Recorded by threshold_analysis.py, which needs the datasets to re-run.
sweep = {
    0.05: (112, 260, 15), 0.10: (108, 148, 19), 0.15: (102, 91, 25),
    0.20: (97, 64, 30),   0.25: (94, 47, 33),   0.30: (92, 36, 35),
    0.35: (90, 28, 37),   0.40: (88, 26, 39),   0.50: (78, 17, 49),
    0.60: (62, 6, 65),
}

print(f"{'conf':>6}{'TP':>6}{'FP':>6}{'FN':>6}{'precision':>11}{'recall':>9}{'F1':>8}")
best = (0.0, 0.0)
for t in SWEEP:
    TP, FP, FN = sweep[t]
    p = TP / (TP + FP)
    r = TP / (TP + FN)
    f1 = 2 * p * r / (p + r)
    if f1 > best[1]:
        best = (t, f1)
    print(f"{t:>6.2f}{TP:>6}{FP:>6}{FN:>6}{p:>11.4f}{r:>9.4f}{f1:>8.4f}")

print(f"\nbest F1 at conf {best[0]:.2f} (F1 {best[1]:.3f})")
print("shipped threshold: 0.50 -> P 0.821, R 0.614, F1 0.703")

**Why 0.5 ships, even though 0.35 scores a better F1.** F1 weights a false
positive and a false negative equally, and here they are not equal. The detector
only picks the metadata filter for retrieval:

* **A miss** (nothing over threshold) falls through to unfiltered retrieval -
  which is exactly the Core Track, already known to work.
* **A false positive** applies the *wrong* boss filter and returns confidently
  sourced pages about a boss that is not on screen. That is worse than no filter
  at all.

Since a miss degrades gracefully and a wrong filter actively misleads, the
threshold is set where precision is high rather than where F1 peaks. At 0.50,
82% of applied filters are correct and 61% of bosses are caught; the other 39%
fall back to unfiltered retrieval. `YOLO_CONFIDENCE_THRESHOLD` is one line in
`.env` if that trade is ever wanted the other way.

### Results

The printed tables above are the authoritative ones; the headline figures are
repeated in the README. Three things stand out beyond the numbers.

**`owl` is by far the strongest class.** On the validation split it reaches mAP50
0.91 against 40 instances, and it is the only class where the model is both
precise and confident. That is what several hundred instances and a consistent
arena backdrop buy. It is also the class whose detections clear 0.5 most often.

**`divine_dragon` is the weakest**, despite going from 21 to 93 instances. That is
the expected cost of the limitation in 5.5.2: its 72 hand-drawn labels all come
from one fight, in one arena, in one session, so the model learned *that*
encounter rather than dragons in general.

**mAP50-95 sits far below mAP50** (roughly 0.39 against 0.74). The boxes are in
the right place, but not tightly drawn. For this application that is the
cheapest possible weakness - the Extended Track only needs the predicted class
name to become a metadata filter, never the box coordinates.

### A trap worth recording: Ultralytics reads numpy as BGR

Mid-evaluation the numbers stopped making sense: `divine_dragon` frames came back
as `owl` with confident boxes, and per-frame confidence was far below what the
mAP implied. The cause was one line, and it was in the backend as well as the
analysis scripts.

**Ultralytics follows the OpenCV convention for array input: a numpy array is
read as BGR and flipped internally. A PIL image is read as RGB and left alone.**
So `model.predict(np.array(rgb_pil_image))` swaps red and blue with no error and
no warning. The model still returns well-placed boxes, just the wrong classes.

Measured on one held-out dragon frame, same pixels:

| Input | Prediction |
|---|---|
| `Image.open(p).convert("RGB")` (PIL) | `divine_dragon` 0.68 |
| `np.array(...)` of that same RGB image | `owl` 0.35 |
| `np.array(...)[:, :, ::-1]` (BGR) | `divine_dragon` 0.68 |

`DetectionService.detect()` now passes the PIL image. The failure mode is worth
knowing about because it is silent and looks like a modelling problem, not a
plumbing one - and it is invisible to `model.val()`, which uses Ultralytics' own
image loader and was correct all along. Only the `predict()` path was affected.

### Honest limitations

Stated here rather than discovered during the demo:

1. **Session diversity is not met for the two classes that need it.** All 275
   hand-drawn frames come from a single 98-minute capture. `corrupted_monk` and
   `divine_dragon` clear the *count* floor but not the *two-session* bar in spec
   5.5.2. Whether the detector generalises to a different playthrough of those
   two fights is untested - the test split cannot answer it, because it inherits
   the same provenance.

2. **The test split is small.** 38 images and 32 instances, with only 2 instances
   each for `corrupted_monk` and `divine_dragon`. On n=2 a single detection
   flipping moves per-class mAP by roughly 0.5, so those two test figures are
   indicative and no more. The validation split carries the per-class story; the
   test split confirms the overall picture is not wildly different.

3. **`guardian_ape` and `genichiro` were never hand-labelled.** They clear the
   floor on the source export alone (97 and 119 instances), so they train fine,
   but their images carry the export's own labelling and none of ours.

4. **The test split comes from the export**, so it measures how well the model
   reproduces the export's labels - not how well it performs on a genuinely new
   playthrough.

5. **The stretched geometry is inherited.** The dataset is 640x640 stretched from
   a 1362x767 stream viewport, so inference stretches the same way. A fullscreen
   screenshot with a different field of view is out of distribution.

6. **Recall at the shipped threshold is 0.61.** Nearly two bosses in five are
   missed and fall back to unfiltered retrieval. That is a deliberate trade, not
   an oversight - see the threshold section above.

The right fix for 1, 3 and 4 is a second recording session, and the pipeline is
built to take one: `extract_frames.py` -> `select_to_label.py` -> `label_tool.py`
-> `build_dataset.py` -> retrain.

### End-to-end sanity check

The metrics above come from `model.val()`. This last cell exercises the path the
*service* actually uses - load the image, stretch it, predict, report the top
class - so the two cannot quietly disagree.

In [ ]:
if WEIGHTS.exists() and DATASET.exists():
    from ultralytics import YOLO
    model = YOLO(str(WEIGHTS))
    images = sorted((DATASET / "test" / "images").glob("*.jpg"))[:10]
    print(f"{'image':<30}{'predicted':<18}{'conf':>7}   truth")
    hits = 0
    for img in images:
        label = DATASET / "test" / "labels" / f"{img.stem}.txt"
        truth = sorted({CLASSES[int(line.split()[0])]
                        for line in label.read_text().splitlines()
                        if len(line.split()) == 5}) if label.exists() else []

        # PIL in, stretched - the same two choices the service makes.
        pil = Image.open(img).convert("RGB").resize((TRAIN_IMGSZ, TRAIN_IMGSZ),
                                                    Image.LANCZOS)
        res = model.predict(pil, verbose=False, conf=CONF_THRESHOLD)[0]
        if res.boxes is not None and len(res.boxes):
            cls = model.names[int(res.boxes.cls[res.boxes.conf.argmax()])]
            conf = float(res.boxes.conf.max())
            ok = cls in truth
            hits += ok
            print(f"{img.stem[:28]:<30}{cls:<18}{conf:>7.3f}   {truth} "
                  f"{'ok' if ok else 'MISS'}")
        else:
            print(f"{img.stem[:28]:<30}{'(below threshold)':<18}{'-':>7}   {truth}")
    print(f"\n{hits}/{len(images)} correct at conf >= {CONF_THRESHOLD}")
else:
    print("needs runs/boss5_v1/weights/best.pt and data/images/dataset5/")

---
## 5.5.8 - Integration with the Core Track

The trained detector is wired into the existing backend, not bolted alongside it.

1. **Weights ship with the repo.** The single `best.pt` the backend loads is
   committed at `backend/data/yolo_model/best.pt` (~6 MB). The `runs/` directory
   it came from stays gitignored - it holds every epoch's checkpoint, the plots
   and the batch previews, all regenerable.
2. **Loaded once at startup**, exactly like the vector store, and reused per
   request. Nothing is retrained at request time.
3. **Failure is never fatal.** No weights, no Ultralytics, no problem - the
   service logs that detection is disabled and the Core Track keeps serving.
   `/health` reports the real state rather than assuming.
4. **`POST /query-image`** runs the detector, and when confidence clears
   `YOLO_CONFIDENCE_THRESHOLD` (0.5) the predicted class name becomes a `boss`
   metadata filter on retrieval. Below the threshold, the request falls through
   to unfiltered retrieval.

**Three details that are easy to get wrong.** All three were hit during this
build, and only the first is obvious in hindsight.

**1. Ultralytics reads a numpy array as BGR.** It follows the OpenCV convention:
array input is taken as BGR and flipped to RGB internally, while PIL input is
taken as RGB and left alone. So `model.predict(np.array(rgb_pil))` swaps red and
blue silently - confident boxes, wrong classes, no warning. `detect()` passes the
PIL image. See the evaluation section for the measurements.

**2. Ultralytics letterboxes by default at predict time.** It scales an uploaded
16:9 screenshot to 640x360 and pads the rest with grey bars, a geometry the model
never saw, because the dataset was *stretched* to square rather than letterboxed.
`detect()` stretches the upload to 640x640 first, which makes the internal
letterbox a no-op.

Measured honestly, this second point is the weaker of the two: across 12 labelled
frames the stretched path scored 9/12 and the default letterboxed path 8/12. That
is a one-frame difference and too small to be conclusive on its own. The stretch
stays because it is the geometry the model was trained on - the right reason, but
not a large measured effect, and it is not claimed as one.

**3. The confidence threshold is a deployment decision, not a default.**
`YOLO_CONFIDENCE_THRESHOLD` is 0.5, chosen for precision over F1 because a wrong
filter misleads while a missing one degrades gracefully.

---
## Summary

| Step | Outcome |
|---|---|
| 5.5.1 | 40 classes -> **5**, the ones the spec names and the only ones with distinct silhouettes |
| 5.5.2 | 1 session, 1920x1080 @ 60 fps, 1h 38m - **limitation stated** |
| 5.5.3 | **1471** frames at 4 fps, 1419 visually distinct |
| 5.5.4 | **275** frames selected, biased to the two starved classes |
| 5.5.5 | Auto-labelling measured at the ceiling (0.463 vs 0.430) and **rejected**; **275** boxes drawn by hand |
| 5.5.5.1 | Merged to **871** images; every class clears 60; imbalance 317:1 -> **3.4:1**; **12** leaked duplicates removed |
| 5.5.6 | Fine-tuned `yolov8n.pt`, 640 px, batch 8, early stopping |
| 5.5.7 | Evaluated on the held-out test split, per class, with limitations stated |
| 5.5.8 | Wired into `POST /query-image`; weights committed; disabled-on-failure |